# 07. POS Classification

**Paper section:** §8.1 POS classification (Table 7).
**What it computes:** Trains n-gram logistic-regression and BiLSTM POS taggers on each language under both Latin-transliteration and Unicode-cuneiform representations, with two tagsets (Universal-style unified tags and the harmonized grammatical-vs-entity split). Sets up the Latin-vs-Unicode comparison that motivates notebook 08.
**Inputs:** Outputs of notebook 01.
**Outputs:** `outputs/table7_pos_classification.csv`.
**Expected runtime (CPU baseline):** ~10 min on CPU; 2-3 min on GPU for BiLSTM.

All randomness uses `SEED = 42`.

## Setup
This notebook uses shared utilities from `_setup.py`:
- `load_corpora()`: returns dict of dataframes per language
- `POS_HARMONIZATION`: dict mapping raw POS tags to unified tagset
- `BASE_PATH`: data directory (set this for your environment)

```python
from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH
```


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except ImportError:
    pass

from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH, set_seeds
set_seeds(42)


In [ ]:
# --- data-availability guard ------------------------------------------
missing = [l for l in ('akk', 'sux', 'elx') if l not in corpora]
if missing:
    print(f'WARNING: {missing} not loaded. Cells that depend on them will be skipped.')
    print(f'Set $CUNEI_DATA to a directory containing alltexts_AKK.csv, alltexts_SUX.csv, and the Elamite files.')
# Convenience: expose datasets dict for cells originally from the monolith.
datasets = {l: corpora[l] for l in ('akk', 'sux', 'elx') if l in corpora}
sign_dict = corpora.get('_sign_dict', {})


In [ ]:
corpora = load_corpora(BASE_PATH)


## Experiment 3: POS Classification

Three classifiers × two representations × two tagsets (unified + grammatical).  
Models: char n-gram Logistic Regression, k-NN on fastText, BiLSTM.

In [ ]:

# ============================================================
# EXPERIMENT 3: POS CLASSIFICATION (LR + k-NN)
# ============================================================
exp3_results = {}
MIN_CLASS_COUNT = 20

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  EXP 3 — {lang.upper()}")
    print(f"{'='*60}")
    model_l = ft_models[lang]['latin']
    model_u = ft_models[lang]['unicode']
    results = {'lang': lang}

    for tagset_name, tag_col in [('unified', 'pos_unified'), ('grammatical', 'pos_grammatical')]:
        print(f"\n  Tagset: {tagset_name}")
        counts = df[tag_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        clf_df = df[df[tag_col].isin(valid)].copy()
        if len(clf_df) > 50000:
            clf_df = clf_df.sample(50000, random_state=SEED)
            print(f"    Subsampled to {len(clf_df):,}")
        if len(valid) < 2 or len(clf_df) < 50:
            print(f"    Insufficient data"); continue
        print(f"    {len(valid)} classes, {len(clf_df):,} tokens")

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

        # Model 1: Char n-gram LR
        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = clf_df[repr_col].astype(str).values
            labels = clf_df[tag_col].values
            try:
                vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
                X = vec.fit_transform(texts)
                clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)
                scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
                results[f'ngram_lr_{tagset_name}_{repr_name}'] = scores.mean()
                print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")
            except Exception as e:
                print(f"    N-gram LR {repr_name} failed: {e}")

        # Model 2: k-NN on fastText
        for repr_name, model, repr_col in [('latin', model_l, 'form_latin'),
                                            ('unicode', model_u, 'form_unicode')]:
            X, y = [], []
            for _, row in clf_df.iterrows():
                word = str(row[repr_col])
                if word and word != 'nan':
                    try: X.append(model.wv[word]); y.append(row[tag_col])
                    except KeyError: continue
            if len(X) < 50: continue
            X, y = np.array(X), np.array(y)
            knn = KNeighborsClassifier(n_neighbors=min(5, len(X)//5))
            try:
                scores = cross_val_score(knn, X, y, cv=cv, scoring='f1_macro')
                results[f'knn_{tagset_name}_{repr_name}'] = scores.mean()
                print(f"    k-NN {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")
            except Exception as e:
                print(f"    k-NN {repr_name} failed: {e}")

    exp3_results[lang] = results

In [ ]:
# ============================================================
# EXPERIMENT 3: BiLSTM (memory-optimized)
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

exp3_bilstm_results = {}
EMBED_DIM = 64; HIDDEN_DIM = 128; N_LAYERS = 2; DROPOUT = 0.3
LR = 0.001; BATCH_SIZE = 64; N_EPOCHS = 30

for lang, df in datasets.items():
    print(f"\n--- BiLSTM: {lang.upper()} ---")
    torch.cuda.empty_cache()

    for pos_col in ['pos_unified', 'pos_grammatical']:
        counts = df[pos_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        clf_df = df[df[pos_col].isin(valid)].copy()
        if len(valid) < 2: continue

        if len(clf_df) > 20000:
            clf_df = clf_df.sample(20000, random_state=SEED)
            print(f"  Subsampled to {len(clf_df):,}")

        le = LabelEncoder()
        labels_enc = le.fit_transform(clf_df[pos_col])
        n_classes = len(le.classes_)

        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = clf_df[repr_col].astype(str).tolist()
            char2idx = {'<pad>': 0, '<unk>': 1}
            for t in texts:
                for ch in t:
                    if ch not in char2idx: char2idx[ch] = len(char2idx)

            max_len = min(max(len(t) for t in texts), 100)
            X = np.zeros((len(texts), max_len), dtype=np.int64)
            for i, t in enumerate(texts):
                for j, ch in enumerate(t[:max_len]):
                    X[i, j] = char2idx.get(ch, 1)

            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
            fold_scores = []
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

            for fold, (train_idx, val_idx) in enumerate(cv.split(X, labels_enc)):
                X_tr = torch.LongTensor(X[train_idx]).to(device)
                y_tr = torch.LongTensor(labels_enc[train_idx]).to(device)
                X_va = torch.LongTensor(X[val_idx]).to(device)
                y_va = labels_enc[val_idx]

                class BiLSTM(nn.Module):
                    def __init__(self):
                        super().__init__()
                        self.embed = nn.Embedding(len(char2idx), EMBED_DIM, padding_idx=0)
                        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, N_LAYERS,
                                           bidirectional=True, batch_first=True, dropout=DROPOUT)
                        self.fc = nn.Linear(HIDDEN_DIM*2, n_classes)
                        self.drop = nn.Dropout(DROPOUT)
                    def forward(self, x):
                        emb = self.drop(self.embed(x))
                        _, (h, _) = self.lstm(emb)
                        h = torch.cat([h[-2], h[-1]], dim=1)
                        return self.fc(self.drop(h))

                model = BiLSTM().to(device)
                opt = torch.optim.Adam(model.parameters(), lr=LR)
                wts = torch.FloatTensor([1.0/max((labels_enc[train_idx]==c).sum(),1) for c in range(n_classes)]).to(device)
                crit = nn.CrossEntropyLoss(weight=wts)

                loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
                model.train()
                for ep in range(N_EPOCHS):
                    for xb, yb in loader:
                        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

                model.eval()
                with torch.no_grad():
                    all_preds = []
                    for k in range(0, len(X_va), 256):
                        all_preds.append(model(X_va[k:k+256]).argmax(dim=1).cpu().numpy())
                    preds = np.concatenate(all_preds)
                fold_scores.append(f1_score(y_va, preds, average='macro'))

                del model, X_tr, y_tr, X_va
                torch.cuda.empty_cache()

            key = f'bilstm_{pos_col}_{repr_name}'
            exp3_bilstm_results.setdefault(lang, {})[key] = np.mean(fold_scores)
            print(f"  {pos_col} {repr_name}: {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")

## Experiment 3b: Separated POS vs NER Analysis

Three sub-tasks to test whether Unicode helps specifically for entity recognition:
1. Grammatical POS only (entities excluded)
2. Entity detection (binary)
3. Entity typing (PN/DN/GN, entities only)


In [ ]:
# ============================================================
# EXPERIMENT 3b: SEPARATED POS vs NER ANALYSIS
# ============================================================
exp3b_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  EXP 3b — {lang.upper()}: Separated POS vs NER")
    print(f"{'='*60}")
    model_l = ft_models[lang]['latin']
    model_u = ft_models[lang]['unicode']
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    results = {'lang': lang}

    # ── Task 1: Grammatical POS only (exclude entities) ──
    gram_df = df[df['ner_tag'] == 'O'].copy()
    counts = gram_df['pos_grammatical'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    gram_clf = gram_df[gram_df['pos_grammatical'].isin(valid)]
    if len(gram_clf) > 50000:
        gram_clf = gram_clf.sample(50000, random_state=SEED)

    if len(valid) >= 2:
        print(f"\n  Task 1: Grammatical POS (entities excluded)")
        print(f"    {len(valid)} classes, {len(gram_clf):,} tokens")
        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = gram_clf[repr_col].astype(str).values
            labels = gram_clf['pos_grammatical'].values
            vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
            X = vec.fit_transform(texts)
            clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
            scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
            results[f'gram_only_{repr_name}'] = scores.mean()
            print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")

    # ── Task 2: Entity detection (binary) ──
    detect_df = df.copy()
    detect_df['is_entity'] = (detect_df['ner_tag'] != 'O').astype(str)
    if len(detect_df) > 50000:
        detect_df = detect_df.sample(50000, random_state=SEED)

    print(f"\n  Task 2: Entity detection (binary)")
    print(f"    {len(detect_df):,} tokens, {(detect_df['is_entity']=='True').mean():.1%} entities")
    for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
        texts = detect_df[repr_col].astype(str).values
        labels = detect_df['is_entity'].values
        vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
        X = vec.fit_transform(texts)
        clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
        scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
        results[f'entity_detect_{repr_name}'] = scores.mean()
        print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")

    # ── Task 3: Entity typing (PN/DN/GN/etc., entities only) ──
    ent_df = df[df['ner_tag'] != 'O'].copy()
    ent_counts = ent_df['ner_tag'].value_counts()
    ent_valid = ent_counts[ent_counts >= MIN_CLASS_COUNT].index.tolist()
    ent_clf = ent_df[ent_df['ner_tag'].isin(ent_valid)]
    if len(ent_clf) > 50000:
        ent_clf = ent_clf.sample(50000, random_state=SEED)

    if len(ent_valid) >= 2:
        print(f"\n  Task 3: Entity typing (entities only)")
        print(f"    {len(ent_valid)} types, {len(ent_clf):,} tokens")
        for repr_name, repr_col in [('latin', 'form_latin'), ('unicode', 'form_unicode')]:
            texts = ent_clf[repr_col].astype(str).values
            labels = ent_clf['ner_tag'].values
            vec = CountVectorizer(analyzer='char', ngram_range=(1,4))
            X = vec.fit_transform(texts)
            clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
            scores = cross_val_score(clf, X, labels, cv=cv, scoring='f1_macro')
            results[f'entity_type_{repr_name}'] = scores.mean()
            print(f"    N-gram LR {repr_name}: {scores.mean():.4f} (±{scores.std():.4f})")

    exp3b_results[lang] = results
